# GWR / EGID Lookup direkt aus Postgres

Dieses Notebook:

1. verbindet sich mit deiner Neon/Postgres-Datenbank über `DATABASE_URL`
2. liest die Adressen aus `listing_details.address`
3. sucht pro Adresse via GeoAdmin/SearchServer die passende Gebäude-Referenz
4. lädt die GWR-Details aus `ch.bfs.gebaeude_wohnungs_register`
5. schreibt die Werte **korrekt in `public.gwr_buildings`**, insbesondere `egid` und `gbaup`

Optional kannst du danach auch noch prüfen, welche Listings keinen Treffer hatten.


In [ ]:
%pip install pandas requests sqlalchemy psycopg2-binary tqdm -q

In [ ]:

import os
import re
import time
from typing import Any, Dict

import pandas as pd
import requests
from sqlalchemy import create_engine, text
from tqdm.auto import tqdm

DB_URL = os.environ.get(
    "DATABASE_URL",
    "postgresql://neondb_owner:npg_X71kIZjKPMcL"
    "@ep-twilight-cloud-agxexq52.c-2.eu-central-1.aws.neon.tech/DSPRO?sslmode=require"
)

print("DB_URL geladen:", DB_URL.split("@")[-1])


## Konfiguration

In [ ]:

SCHEMA = "public"
LISTING_DETAILS_TABLE = "listing_details"
TARGET_TABLE = "gwr_buildings"

ADDRESS_COLUMN = "address"
LISTING_ID_COLUMN = "listing_id"

REQUESTS_PER_SECOND = 3
DELAY = 1.0 / REQUESTS_PER_SECOND

API_SEARCH_URL = "https://api3.geo.admin.ch/rest/services/api/SearchServer"
API_BASE = "https://api3.geo.admin.ch"

print(f"Zieltable: {SCHEMA}.{TARGET_TABLE}")
print(f"Quelltable: {SCHEMA}.{LISTING_DETAILS_TABLE}")
print(f"Adressspalte: {ADDRESS_COLUMN}")


## Datenbankverbindung

In [ ]:

engine = create_engine(DB_URL, pool_pre_ping=True)

with engine.connect() as conn:
    version = conn.execute(text("select version()")).scalar()
    print("✅ Verbunden")
    print(version[:120] + "...")


## Listings mit Adresse laden

Es wird bewusst aus `listing_details.address` gelesen, weil du gesagt hast:  
**„Adresse ist im Listing adress“**.


In [ ]:

query = text(f"""
    SELECT
        ld.{LISTING_ID_COLUMN} AS listing_id,
        ld.{ADDRESS_COLUMN} AS address
    FROM {SCHEMA}.{LISTING_DETAILS_TABLE} ld
    WHERE ld.{ADDRESS_COLUMN} IS NOT NULL
      AND btrim(ld.{ADDRESS_COLUMN}) <> ''
    ORDER BY ld.{LISTING_ID_COLUMN}
""")

listing_df = pd.read_sql(query, engine)

print("Geladene Zeilen:", len(listing_df))
display(listing_df.head(10))


In [ ]:

def normalize_address(addr: Any) -> str:
    if not isinstance(addr, str):
        return ""
    addr = re.sub(r"\s+", " ", addr.strip())
    return addr

listing_df["address_norm"] = listing_df["address"].apply(normalize_address)
listing_df = listing_df[listing_df["address_norm"] != ""].copy()

print("Nach Normalisierung:", len(listing_df))
display(listing_df.head(10))


## GeoAdmin / GWR Helper

In [ ]:

def search_address(address: str, session: requests.Session) -> Dict[str, Any]:
    params = {
        "searchText": address,
        "type": "locations",
        "origins": "address",
        "limit": 1,
        "returnGeometry": "false",
    }
    resp = session.get(API_SEARCH_URL, params=params, timeout=20)
    resp.raise_for_status()
    data = resp.json()

    results = data.get("results", [])
    if not results:
        return {"status": "not_found", "address": address}

    best = results[0]
    attrs = best.get("attrs", {}) or {}

    gwr_link = None
    for link in attrs.get("links", []) or []:
        if link.get("title") == "ch.bfs.gebaeude_wohnungs_register":
            href = link.get("href")
            if href:
                gwr_link = href
                break

    feature_id = attrs.get("featureId") or attrs.get("feature_id")
    egid_from_feature = None
    if feature_id:
        egid_from_feature = str(feature_id).split("_")[0]

    return {
        "status": "found",
        "address": address,
        "match_label": (attrs.get("label") or "").replace("<b>", "").replace("</b>", ""),
        "score": best.get("weight"),
        "origin": attrs.get("origin"),
        "feature_id": feature_id,
        "egid_from_feature": egid_from_feature,
        "gwr_link": gwr_link,
    }


def fetch_gwr_feature(gwr_href: str, session: requests.Session) -> Dict[str, Any]:
    url = gwr_href if gwr_href.startswith("http") else f"{API_BASE}{gwr_href}"
    params = {"returnGeometry": "false"}
    resp = session.get(url, params=params, timeout=20)
    resp.raise_for_status()
    data = resp.json()

    feature = data.get("feature", {}) or {}
    attrs = feature.get("attributes", {}) or {}

    attrs = {str(k).lower(): v for k, v in attrs.items()}
    attrs["_source_url"] = url
    attrs["_feature_id"] = feature.get("featureId") or feature.get("id")
    return attrs


## Lookup für alle eindeutigen Adressen

In [ ]:

unique_addresses = (
    listing_df[["address_norm"]]
    .drop_duplicates()
    .rename(columns={"address_norm": "address"})
    .reset_index(drop=True)
)

print("Eindeutige Adressen:", len(unique_addresses))

lookup_rows = []
errors = []

with requests.Session() as session:
    session.headers.update({"User-Agent": "GWR-EGID-DB-Sync/1.0"})

    for address in tqdm(unique_addresses["address"], desc="Adresse → EGID/GWR"):
        try:
            search_hit = search_address(address, session)

            if search_hit["status"] != "found":
                lookup_rows.append(search_hit)
                time.sleep(DELAY)
                continue

            gwr_data = {}
            if search_hit.get("gwr_link"):
                gwr_data = fetch_gwr_feature(search_hit["gwr_link"], session)

            row = {**search_hit, **gwr_data}

            if not row.get("egid") and row.get("egid_from_feature"):
                row["egid"] = row["egid_from_feature"]

            lookup_rows.append(row)

        except Exception as e:
            lookup_rows.append({
                "status": f"error: {type(e).__name__}: {str(e)[:180]}",
                "address": address,
            })
            errors.append(address)

        time.sleep(DELAY)

lookup_df = pd.DataFrame(lookup_rows)
print("Lookup fertig:", len(lookup_df))
print("Fehler:", len(errors))
display(lookup_df.head(10))


## Für Insert vorbereiten

Hier werden nur Spalten verwendet, die tatsächlich in `gwr_buildings` existieren.
So wird nichts in eine falsche Spalte geschrieben.


In [ ]:

with engine.connect() as conn:
    col_query = text("""
        SELECT column_name
        FROM information_schema.columns
        WHERE table_schema = :schema
          AND table_name = :table
        ORDER BY ordinal_position
    """)
    target_columns = [r[0] for r in conn.execute(col_query, {"schema": SCHEMA, "table": TARGET_TABLE})]

print("Spalten in gwr_buildings:")
print(target_columns)


In [ ]:

required = ["egid", "gbaup"]
missing_required = [c for c in required if c not in target_columns]
if missing_required:
    raise ValueError(f"Pflichtspalten fehlen in {TARGET_TABLE}: {missing_required}")

good_df = lookup_df[lookup_df["egid"].notna()].copy()

for col in ["egid", "gbaup"]:
    if col in good_df.columns:
        good_df[col] = pd.to_numeric(good_df[col], errors="coerce").astype("Int64")

insertable_cols = [c for c in good_df.columns if c in target_columns]
db_df = good_df[insertable_cols].copy()
db_df = db_df[db_df["egid"].notna()].drop_duplicates(subset=["egid"]).reset_index(drop=True)

print("Zeilen bereit für DB:", len(db_df))
print("Spalten für Insert:", insertable_cols)
display(db_df.head(10))


## Upsert in `gwr_buildings`

Annahme: `egid` ist der natürliche Schlüssel.  
Falls `egid` noch **kein Unique/Primary Key** ist, bekommst du hier einen Fehler. Dann unten die Diagnose-Zelle ausführen.


In [ ]:

NUMERIC_INT_COLS = {
    "egid", "gbaup", "gbauj", "gbaum", "gabbj", "garea", "gvol", "gvolnorm", "gvolsce",
    "gastw", "ganzwhg", "gazzi", "gschutzr", "gebf", "gkode", "gkodn", "gksce",
    "gstat", "gkat", "gklas", "gwaerzh1", "genh1", "gwaersceh1", "gwaerzh2", "genh2",
    "gwaersceh2", "gwaerzw1", "genw1", "gwaerscew1", "gwaerzw2"
}

def upsert_gwr_buildings(df: pd.DataFrame, engine, schema: str, table: str):
    if df.empty:
        print("Nichts zu schreiben.")
        return 0

    cols = list(df.columns)
    placeholders = ", ".join([f":{c}" for c in cols])
    quoted_cols = ", ".join(cols)

    update_cols = [c for c in cols if c != "egid"]
    set_clause = ", ".join([f"{c} = EXCLUDED.{c}" for c in update_cols])

    sql = text(f"""
        INSERT INTO {schema}.{table} ({quoted_cols})
        VALUES ({placeholders})
        ON CONFLICT (egid) DO UPDATE
        SET {set_clause}
    """)

    payload = []
    for rec in df.to_dict("records"):
        clean = {}
        for k, v in rec.items():
            if pd.isna(v):
                clean[k] = None
            elif k in NUMERIC_INT_COLS:
                clean[k] = int(v)
            else:
                clean[k] = v
        payload.append(clean)

    with engine.begin() as conn:
        conn.execute(sql, payload)

    return len(payload)

written = upsert_gwr_buildings(db_df, engine, SCHEMA, TARGET_TABLE)
print(f"✅ In gwr_buildings geschrieben/aktualisiert: {written}")


## Kontrolle in der DB

In [ ]:

check_query = text(f"""
    SELECT egid, gbaup
    FROM {SCHEMA}.{TARGET_TABLE}
    WHERE egid IS NOT NULL
    ORDER BY egid
    LIMIT 20
""")

check_df = pd.read_sql(check_query, engine)
display(check_df)


## Optional: Diagnose, falls `ON CONFLICT (egid)` fehlschlägt

In [ ]:

diag_sql = text("""
SELECT
    tc.constraint_name,
    tc.constraint_type,
    kcu.column_name
FROM information_schema.table_constraints tc
JOIN information_schema.key_column_usage kcu
  ON tc.constraint_name = kcu.constraint_name
 AND tc.table_schema = kcu.table_schema
WHERE tc.table_schema = :schema
  AND tc.table_name = :table
ORDER BY tc.constraint_type, tc.constraint_name, kcu.ordinal_position
""")

with engine.connect() as conn:
    constraints_df = pd.read_sql(diag_sql, conn, params={"schema": SCHEMA, "table": TARGET_TABLE})

display(constraints_df)
